In [0]:
import os
current_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
print(current_user)

In [0]:
import os
import re

CONFIG_DIR = "/Workspace/Users/jeremi.santoso@metrodata.co.id/dbx-pipeline-integration/config/"

def get_all_table_names(config_dir: str) -> list[str]:
    files = os.listdir(config_dir)
    table_names = []
    for f in files:
        match = re.match(r"table_(.+)\.yaml$", f)
        if match:
            table_names.append(match.group(1))
    return sorted(table_names)

TABLE_LIST = get_all_table_names(CONFIG_DIR)
print("Table names:", TABLE_LIST)
print("Total table:", len(TABLE_LIST))

In [0]:
%sql
SELECT
    origin.flow_name AS table_name,
    origin.pipeline_name,
    timestamp,
    details:flow_progress.status AS status,
    TRY_CAST(details:flow_progress.metrics.num_output_rows AS BIGINT) AS num_output_rows,
    TRY_CAST(details:flow_progress.metrics.num_upserted_rows AS BIGINT) AS num_upserted_rows,
    TRY_CAST(details:flow_progress.metrics.num_deleted_rows AS BIGINT) AS num_deleted_rows,
    TRY_CAST(details:flow_progress.data_quality.dropped_records AS BIGINT) AS dropped_records
FROM event_log('344fcd2c-5048-43f7-8d51-474fc37564eb')
WHERE event_type = 'flow_progress'
  AND origin.flow_name IS NOT NULL
  AND origin.flow_name != 'pipelines.flowTimeMetrics.missingFlowName'
ORDER BY timestamp DESC
LIMIT 20

In [0]:
%sql
SELECT *
FROM event_log('344fcd2c-5048-43f7-8d51-474fc37564eb')
-- WHERE event_type = 'flow_progress'
-- ORDER BY timestamp DESC
-- LIMIT 3

In [0]:
%sql
SELECT
  t.pipeline_id,
  p.name AS pipeline_name,
  t.update_id,
  t.result_state,
  t.trigger_type,
  t.period_start_time,
  t.period_end_time,
  TIMESTAMPDIFF(SECOND, t.period_start_time, t.period_end_time) AS duration_seconds
FROM system.lakeflow.pipeline_update_timeline t
LEFT JOIN (
  SELECT *, ROW_NUMBER() OVER(PARTITION BY workspace_id, pipeline_id ORDER BY change_time DESC) AS rn
  FROM system.lakeflow.pipelines
) p ON t.pipeline_id = p.pipeline_id AND t.workspace_id = p.workspace_id AND p.rn = 1
WHERE t.period_start_time > CURRENT_TIMESTAMP() - INTERVAL 30 DAYS
ORDER BY t.period_start_time DESC

In [0]:
%sql
select * 
FROM system.lakeflow.pipeline_update_timeline
where result_state = "COMPLETED"

In [0]:
%sql
SELECT
  origin.flow_name AS table_name,
  timestamp,
  details:flow_progress.status AS status,
  details:flow_progress.metrics AS metrics
FROM
  event_log('344fcd2c-5048-43f7-8d51-474fc37564eb')
WHERE
  event_type = 'flow_progress'
  AND details:flow_progress.metrics IS NOT NULL
ORDER BY
  timestamp DESC
LIMIT 10


In [0]:
from pyspark.sql.functions import col

df = spark.sql("DESCRIBE HISTORY mii_workspace.default.bronze_s3_customer")
display(
    df.select(
        "version",
        "operation",
        "timestamp",
        col("operationMetrics")["numOutputRows"].alias("num_output_rows"),
        col("operationMetrics")["numFiles"].alias("num_files")
    )
)

In [0]:
from pyspark.sql.functions import col
from functools import reduce
from pyspark.sql import DataFrame

# Get row counts from DESCRIBE HISTORY for each pipeline table
tables = [
    "mii_workspace.default.bronze_s3_customer",
    "mii_workspace.default.bronze_s3_employee",
    "mii_workspace.default.silver_s3_customer"
]

dfs = []
for table in tables:
    df = spark.sql(f"DESCRIBE HISTORY {table}") \
        .filter(col("operation") == "STREAMING UPDATE") \
        .select(
            col("operationMetrics")["numOutputRows"].alias("num_output_rows"),
            "timestamp",
            "clusterId"
        ).withColumn("table_name", lit(table))
    dfs.append(df)

from pyspark.sql.functions import lit
# Rebuild with lit import
dfs = []
for table in tables:
    df = spark.sql(f"DESCRIBE HISTORY {table}") \
        .filter(col("operation") == "STREAMING UPDATE") \
        .select(
            lit(table).alias("table_name"),
            col("operationMetrics")["numOutputRows"].cast("long").alias("num_output_rows"),
            "timestamp"
        )
    dfs.append(df)

history_df = reduce(DataFrame.unionAll, dfs)

# Join to pipeline_update_timeline by timestamp range
pipeline_updates = spark.sql("""
    SELECT pipeline_id, update_id, period_start_time, period_end_time
    FROM system.lakeflow.pipeline_update_timeline
    WHERE pipeline_id = '344fcd2c-5048-43f7-8d51-474fc37564eb'
      AND result_state = 'COMPLETED'
""")

result = history_df.join(
    pipeline_updates,
    (history_df.timestamp >= pipeline_updates.period_start_time) &
    (history_df.timestamp <= pipeline_updates.period_end_time),
    "left"
).select(
    "pipeline_id", "update_id", "table_name", "num_output_rows",
    history_df.timestamp.alias("write_timestamp"),
    "period_start_time", "period_end_time"
).orderBy("write_timestamp", ascending=False)

display(result)